# Ngày 1 · NumPy cho Machine Learning

- Đọc shape từ phải sang trái
- Chuẩn hóa từng cột và vector hóa khoảng cách

**Đọc:**
- Vũ Hữu Tiệp, Machine Learning cơ bản, Chương 1 (Ôn tập Đại số tuyến tính), mục 1.1–1.3
- Harris et al. (2020), Array programming with NumPy, Nature 585 — mục về broadcasting
- UC Berkeley CS189, Lecture 02 "Data Tools"

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge

DATA = next((p for p in (Path('data'), Path('../data')) if (p/'real_estate.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Mở notebook mẫu từ thư mục py30 hoặc bai_lam.')
real_estate = pd.read_csv(DATA / 'real_estate.csv')

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
import ast, inspect
_A = np.array([[1., 2.], [-3., 4.]])
_B = np.array([[2., 2.], [7., 0.], [0., 0.]])
_loop_dist = np.array([[sum((a - b)**2) for b in _B] for a in _A])
def _preserves_input(fn):
    x = np.array([[2., 5.], [4., 5.]])
    original = x.copy()
    fn(x)
    return np.array_equal(x, original)
def _no_loops(fn):
    forbidden = (ast.For, ast.While, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    return not any(isinstance(node, forbidden) for node in ast.walk(ast.parse(inspect.getsource(fn))))

_R = np.random.default_rng(0).normal([3., -2., 10.], [2., .5, 7.], size=(50, 3))


## Chuẩn hóa từng cột

Gợi ý: Dùng X.std(axis=0) (ddof=0). Thay mẫu số 0 bằng 1 trước khi chia.

In [ ]:
def standardize(X):
    raise NotImplementedError


In [ ]:
_report('d1-standardize',
    _check('Mỗi cột có trung bình 0, độ lệch chuẩn 1', lambda: (np.allclose(standardize(_R).mean(0), 0) and np.allclose(standardize(_R).std(0), 1), 'Dữ liệu ngẫu nhiên 50×3, độ lệch chuẩn quần thể (ddof=0).')),
    _check('Cột hằng về 0', lambda: (np.allclose(standardize(np.array([[2., 5.], [4., 5.]])), [[-1, 0], [1, 0]]), 'Cột 1 thay đổi, cột 2 hằng.')),
    _check('Không sửa dữ liệu đầu vào', lambda: (_preserves_input(standardize), 'So sánh X trước và sau khi gọi hàm.')),
)


## Khoảng cách bình phương giữa hai nhóm

Gợi ý: Dùng A[:, None, :] và B[None, :, :], rồi giảm chiều đặc trưng.

In [ ]:
def pairwise_sq_dists(A, B):
    raise NotImplementedError


In [ ]:
_report('d1-distance',
    _check('Khớp vòng lặp trên hai nhóm khác kích thước', lambda: (np.allclose(pairwise_sq_dists(_A, _B), _loop_dist), 'Một hàng của A ứng với mọi hàng của B.')),
    _check('Vector hóa trong thân hàm', lambda: (_no_loops(pairwise_sq_dists), 'Tránh các vòng lặp và biểu thức comprehension.')),
)


## Dự án

Khảo sát cột số trên UCI Real Estate. Vẽ phân bố giá và ghi đơn vị 10.000 TWD/ping. Sau đó tải House Prices và so danh sách cột số, định danh và mục tiêu.